## Cell 1 · 패키지 설치

In [ ]:
# ============================================================
# Cell 1: 패키지 설치
# - qai-hub-models[facemap-3dmm-quantized] : Qualcomm 공식 패키지
#   from_pretrained()으로 PyTorch 모델을 로컬 다운로드
# - insightface + onnxruntime : bbox 검출
# ============================================================
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Qualcomm 공식 패키지 (v0.49.1 고정)
pip("qai-hub-models[facemap-3dmm-quantized]==0.49.1")

# InsightFace 얼굴 검출
pip("insightface", "onnxruntime")

# 기타
pip("opencv-python-headless", "matplotlib", "Pillow", "scipy")

print("✅ 설치 완료")


✅ 설치 완료


## Cell 2 · 임포트 & 전역 설정

In [ ]:
# ============================================================
# Cell 2: 임포트 & 전역 변수
# RESULTS dict — 마지막 셀 전까지 파일 저장 금지
# ============================================================
import io, json, os, warnings
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from insightface.app import FaceAnalysis

warnings.filterwarnings("ignore")
np.random.seed(42)

# 결과 버퍼 (마지막 셀에서만 파일로 저장)
RESULTS = {
    "coeff": None,          # (265,) float32
    "landmarks68": None,    # (68, 2) 원본 이미지 좌표
    "blendshape_52": None,  # dict {name: float}
    "landmark_vis_img": None,
    "blendshape_vis_img": None,
    "original_img": None,
    "bbox": None,
    "is_pytorch_mode": True,  # PyTorch 로컬 추론 여부
}

print("✅ 임포트 완료")
print(f"   PyTorch: {torch.__version__}")
print(f"   NumPy  : {np.__version__}")


✅ 임포트 완료
   PyTorch: 2.8.0+cu128
   NumPy  : 2.0.2


## Cell 3 · Qualcomm facemap_3dmm 모델 로드

In [ ]:
# ============================================================
# Cell 3: Qualcomm facemap_3dmm 모델 로드
#
# [공식 설치 방법]
#   pip install "qai-hub-models[facemap-3dmm-quantized]"
#   ref: https://github.com/qualcomm/ai-hub-models/tree/v0.49.1
#        /qai_hub_models/models/facemap_3dmm
#
# [로드 방법]
#   from qai_hub_models.models.facemap_3dmm_quantized import Model
#   model = Model.from_pretrained()  → PyTorch fp32 가중치 다운로드
#
# [추론 모드]
#   eval-mode=fp  : PyTorch 로컬 추론 (AI Hub 계정 불필요)
#   eval-mode=on-device : Qualcomm 디바이스 필요 (Colab 불가)
#
# [모델 출력]
#   - 입력 : (1, 3, 128, 128) float32 (0~1 정규화)
#   - 출력 : (1, 265) float32 coefficient
#     coeff[0:80]   identity (shape PCA)
#     coeff[80:144] expression PCA
#     coeff[144:147] rotation (Rodrigues 3D)
#     coeff[147:150] translation (tx, ty, tz)
#     coeff[150:151] focal / scale
#     coeff[151:265] 미사용 (텍스처/조명)
# ============================================================

from qai_hub_models.models.facemap_3dmm_quantized import Model as FaceMap3DMMModel

print("모델 다운로드 & 로드 중... (첫 실행 시 수십 초 소요)")
facemap_model = FaceMap3DMMModel.from_pretrained()
facemap_model.eval()

print(f"✅ 모델 로드 완료")
print(f"   타입  : {type(facemap_model)}")

# 입출력 스펙 확인
try:
    spec = facemap_model.get_input_spec()
    print(f"   입력 스펙: {spec}")
except Exception:
    print("   입력 스펙: (1, 3, 128, 128) float32  [표준 스펙]")


ModuleNotFoundError: No module named 'qai_hub_models.models.facemap_3dmm_quantized'

## Cell 4 · InsightFace 초기화

In [ ]:
# ============================================================
# Cell 4: InsightFace FaceAnalysis 초기화
#
# buffalo_l 패키지:
#   - 검출기 : RetinaFace (det_size=640×640)
#   - 인식기 : ArcFace
#   모델은 ~/.insightface/models/ 에 자동 다운로드
# ============================================================

face_app = FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640))

print("✅ InsightFace 초기화 완료")
print("   검출기 : RetinaFace  (buffalo_l)")
print("   입력   : 640×640")


## Cell 5 · 유틸리티 함수 정의

In [ ]:
# ============================================================
# Cell 5: 유틸리티 함수
# ============================================================

# ── coeff 분할 ────────────────────────────────────────────────
def split_coeff(coeff: np.ndarray) -> dict:
    """
    265차원 coeff를 구간별로 분할.
    [공식 근거] Qualcomm facemap_3dmm v0.49.1 출력 구조.
    """
    assert len(coeff) == 265, f"265차원 필요, 실제: {len(coeff)}"
    return {
        "identity":    coeff[0:80],
        "expression":  coeff[80:144],
        "rotation":    coeff[144:147],
        "translation": coeff[147:150],
        "focal":       coeff[150:151],
        "unused":      coeff[151:265],
    }


# ── 전처리: bbox crop → 모델 입력 텐서 ─────────────────────────
def preprocess_for_facemap(
    img_bgr: np.ndarray,
    bbox: tuple,
    size: int = 128
) -> torch.Tensor:
    """
    InsightFace bbox → facemap_3dmm 입력 텐서.

    [공식 근거] facemap_3dmm 입력 규격:
      - 128×128 RGB
      - float32, [0, 1] 정규화
      - shape: (1, 3, 128, 128)

    [bbox 좌표 일관성]
      - InsightFace bbox를 그대로 사용 (int 변환 + clip)
      - landmark 역변환 시에도 동일 bbox 사용 → 좌표계 일치 보장
    """
    x1, y1, x2, y2 = [int(v) for v in bbox]
    h, w = img_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    crop = img_bgr[y1:y2, x1:x2]
    crop = cv2.resize(crop, (size, size), interpolation=cv2.INTER_LINEAR)
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)           # (128,128,3) uint8
    tensor = torch.from_numpy(crop_rgb).float() / 255.0        # [0,1]
    tensor = tensor.permute(2, 0, 1).unsqueeze(0)              # (1,3,128,128)
    return tensor, crop_rgb


# ── PyTorch 추론 ──────────────────────────────────────────────
@torch.no_grad()
def run_facemap(model, img_tensor: torch.Tensor) -> np.ndarray:
    """
    facemap_3dmm PyTorch 로컬 추론.
    반환: (265,) float32 ndarray
    """
    out = model(img_tensor)
    # 출력이 tuple/list일 수 있음
    if isinstance(out, (tuple, list)):
        out = out[0]
    coeff = out.squeeze().cpu().numpy()  # (265,)
    return coeff.astype(np.float32)


# ── Rodrigues → 회전 행렬 ────────────────────────────────────
def rodrigues_to_R(rvec: np.ndarray) -> np.ndarray:
    rvec = rvec.reshape(3, 1).astype(np.float64)
    R, _ = cv2.Rodrigues(rvec)
    return R.astype(np.float32)


# ── 68 Landmark 복원 (crop 128×128 기준) ─────────────────────
def reconstruct_landmarks68_from_coeff(
    coeff: np.ndarray,
    mean_face_path: str = None
) -> np.ndarray:
    """
    3DMM coeff → 68개 2D landmark (crop 좌표계, 128×128).

    [공식 근거] Qualcomm 후처리:
      shape_3d = meanFace + shapeBasis @ id + exprBasis @ expr
      pts_2d   = (R[:2] @ shape_3d.T + t[:2]).T
      px       = pts_2d * focal + image_center

    [basis 파일 없을 경우]
      qai_hub_models 패키지 내부 리소스에서 로드 시도.
      실패 시 coeff의 rotation/translation/focal로만 투영.

    반환: (68, 2) float32 (crop 128×128 기준 픽셀 좌표)
    """
    c = split_coeff(coeff)
    R = rodrigues_to_R(c["rotation"])
    t = c["translation"]
    f = float(c["focal"][0])
    if abs(f) < 1e-4:
        f = 1.0

    # qai_hub_models 패키지 내 basis 파일 탐색
    import importlib.resources as ir
    mean_face = None
    shape_basis = None
    expr_basis  = None

    try:
        import qai_hub_models.models.facemap_3dmm as fm_pkg
        pkg_dir = Path(fm_pkg.__file__).parent
        candidates = [
            pkg_dir / "utils" / "mean_face.npy",
            pkg_dir / "mean_face.npy",
        ]
        for p in candidates:
            if p.exists():
                mean_face = np.load(p).reshape(-1)
                print(f"  meanFace 로드: {p}")
                break

        shape_candidates = [
            pkg_dir / "utils" / "shape_basis.npy",
            pkg_dir / "shape_basis.npy",
        ]
        for p in shape_candidates:
            if p.exists():
                shape_basis = np.load(p)   # (204, 80)
                print(f"  shapeBasis 로드: {p}")
                break

        expr_candidates = [
            pkg_dir / "utils" / "expression_basis.npy",
            pkg_dir / "expression_basis.npy",
        ]
        for p in expr_candidates:
            if p.exists():
                expr_basis = np.load(p)    # (204, 64)
                print(f"  exprBasis 로드: {p}")
                break

    except Exception as e:
        print(f"  basis 탐색 실패: {e}")

    # basis가 있으면 3D 형상 재구성
    if mean_face is not None:
        mf = mean_face.reshape(-1)[:204] if len(mean_face) >= 204 else np.zeros(204, np.float32)
        sc = shape_basis @ c["identity"] if shape_basis is not None and shape_basis.shape == (204, 80) else np.zeros(204)
        ec = expr_basis  @ c["expression"] if expr_basis is not None and expr_basis.shape == (204, 64) else np.zeros(204)
        shape_3d = (mf + sc + ec).reshape(68, 3)
    else:
        # [근사] basis 없이 단위 구에 투영
        # 68개 랜드마크 위치를 identity/expression coeff로 근사 생성
        # ⚠️ 이 블록은 basis 파일 없을 때만 사용하는 근사입니다.
        print("  ⚠️  basis 파일 없음 → 근사 3D 형상 사용 (landmark 부정확)")
        # dlib 68-point 표준 정규화 위치 (논문 기반 근사)
        theta = np.linspace(0, 2*np.pi, 17, endpoint=False)
        jaw_x = 0.45 * np.cos(theta - np.pi/2)
        jaw_y = 0.45 * np.sin(theta - np.pi/2)
        # 대략적인 68점 배치 (정규화 -0.5~0.5 범위)
        base_pts = np.array([
            *zip(jaw_x, jaw_y, [0]*17),  # 0-16: jaw
            *[(-0.30+i*0.06, -0.28, 0) for i in range(5)],  # 17-21: left brow
            *[(0.06+i*0.06,  -0.28, 0) for i in range(5)],  # 22-26: right brow
            *([(0.0, -0.15+i*0.06, 0) for i in range(4)]    # 27-30: nose bridge
              +[(j*0.08-0.16, -0.01, 0) for j in range(5)]),# 31-35: nose base
            *[(-0.23+i*0.05, -0.10, 0.05) for i in range(6)],  # 36-41: left eye
            *[( 0.03+i*0.05, -0.10, 0.05) for i in range(6)],  # 42-47: right eye
            *[(-0.20+i*0.057, 0.15, 0) for i in range(8)],  # 48-55: outer lip
            *[(-0.16+i*0.057, 0.18, 0) for i in range(4)],  # 56-59: outer lip btm
            *[(-0.13+i*0.065, 0.14, 0) for i in range(4)],  # 60-63: inner lip
            *[(-0.10+i*0.065, 0.17, 0) for i in range(4)],  # 64-67: inner lip btm
        ], dtype=np.float32)
        if len(base_pts) < 68:
            base_pts = np.vstack([base_pts, np.zeros((68-len(base_pts), 3), np.float32)])
        shape_3d = base_pts[:68]

    # 2D 투영 (crop 128×128 기준)
    projected = (R[:2, :] @ shape_3d.T).T + t[:2]  # (68, 2)
    center = np.array([64.0, 64.0])
    scale  = 64.0  # 128/2
    lmks_crop = projected * f * scale + center
    return lmks_crop.astype(np.float32)


# ── crop → 원본 좌표 역변환 ──────────────────────────────────
def crop_to_original(
    lmks_crop: np.ndarray,
    bbox: tuple,
    crop_size: int = 128
) -> np.ndarray:
    """
    [공식 근거]
    crop = img[y1:y2, x1:x2] → resize to crop_size
    역변환 = lmk * (bbox_wh / crop_size) + bbox_topleft

    InsightFace bbox와 동일 bbox를 사용하므로 좌표계 일치.
    """
    x1, y1, x2, y2 = [int(v) for v in bbox]
    sx = (x2 - x1) / crop_size
    sy = (y2 - y1) / crop_size
    out = lmks_crop.copy()
    out[:, 0] = lmks_crop[:, 0] * sx + x1
    out[:, 1] = lmks_crop[:, 1] * sy + y1
    return out.astype(np.float32)


print("✅ 유틸리티 함수 정의 완료")


## Cell 6 · 52 Blendshape 변환 함수

In [ ]:
# ============================================================
# Cell 6: 52 Blendshape 변환
#
# [중요 전제]
# Qualcomm facemap_3dmm → ARKit/MediaPipe 52 blendshape의
# 공식 direct mapping은 존재하지 않습니다.
#
# 구성:
#  A) [공식 근거] landmark 거리 기반 (EAR, 입 열림 비율 등)
#  B) [근사]     expression PCA coeff 첫 N개를 보조 신호로 활용
#  C) [미구현]   눈 시선(eyeLookIn/Out) → 홍채 검출 필요, 0으로 설정
# ============================================================

# MediaPipe/ARKit 52 blendshape 표준 명칭
BLENDSHAPE_NAMES = [
    "browDownLeft",     "browDownRight",    "browInnerUp",
    "browOuterUpLeft",  "browOuterUpRight",
    "cheekPuff",        "cheekSquintLeft",  "cheekSquintRight",
    "eyeBlinkLeft",     "eyeBlinkRight",
    "eyeLookDownLeft",  "eyeLookDownRight",
    "eyeLookInLeft",    "eyeLookInRight",
    "eyeLookOutLeft",   "eyeLookOutRight",
    "eyeLookUpLeft",    "eyeLookUpRight",
    "eyeSquintLeft",    "eyeSquintRight",
    "eyeWideLeft",      "eyeWideRight",
    "jawForward",       "jawLeft",          "jawOpen",      "jawRight",
    "lipsUpperClose",   "lipsLowerClose",
    "mouthClose",
    "mouthDimpleLeft",  "mouthDimpleRight",
    "mouthFrownLeft",   "mouthFrownRight",
    "mouthFunnel",
    "mouthLeft",
    "mouthLowerDownLeft",  "mouthLowerDownRight",
    "mouthPressLeft",   "mouthPressRight",
    "mouthPucker",
    "mouthRight",
    "mouthRollLower",   "mouthRollUpper",
    "mouthShrugLower",  "mouthShrugUpper",
    "mouthSmileLeft",   "mouthSmileRight",
    "mouthStretchLeft", "mouthStretchRight",
    "mouthUpperUpLeft", "mouthUpperUpRight",
    "noseSneerLeft",
]
assert len(BLENDSHAPE_NAMES) == 52


def _norm(v, lo, hi):
    """[lo,hi] → [0,1] 클리핑 정규화"""
    return float(np.clip((v - lo) / (max(hi - lo, 1e-6)), 0.0, 1.0))


def _lmk_blendshapes(lmks: np.ndarray) -> dict:
    """
    68 landmark 기하학 → blendshape 근사.
    [근사/규칙 기반] 이 함수 전체는 landmark 거리비율 기반이며
    ARKit 공식 알고리즘이 아닙니다.

    dlib 68-point 인덱스:
      0-16  jaw
      17-21 left brow,  22-26 right brow
      27-35 nose
      36-41 left eye,   42-47 right eye
      48-67 mouth
    """
    bs = {}
    fw = np.linalg.norm(lmks[0] - lmks[16]) + 1e-6  # 얼굴 폭 기준

    # ── EAR (Eye Aspect Ratio) 기반 눈 ──────────────────────────
    # [공식 근거] Soukupova & Cech (2016) EAR = (A+B)/(2C)
    def ear(p1,p2,p3,p4,p5,p6):
        return (np.linalg.norm(p2-p6)+np.linalg.norm(p3-p5))/(2*np.linalg.norm(p1-p4)+1e-6)
    el = ear(lmks[36],lmks[37],lmks[38],lmks[39],lmks[40],lmks[41])
    er = ear(lmks[42],lmks[43],lmks[44],lmks[45],lmks[46],lmks[47])
    bs["eyeBlinkLeft"]   = _norm(0.32-el, 0.0, 0.32)
    bs["eyeBlinkRight"]  = _norm(0.32-er, 0.0, 0.32)
    bs["eyeWideLeft"]    = _norm(el-0.34, 0.0, 0.12)
    bs["eyeWideRight"]   = _norm(er-0.34, 0.0, 0.12)
    bs["eyeSquintLeft"]  = bs["eyeBlinkLeft"]  * 0.5
    bs["eyeSquintRight"] = bs["eyeBlinkRight"] * 0.5

    # ── 눈 시선 [미구현] ─────────────────────────────────────────
    # 홍채 중심 없이 시선 방향 추정 불가 → 0 설정
    for k in ["eyeLookInLeft","eyeLookInRight","eyeLookOutLeft","eyeLookOutRight"]:
        bs[k] = 0.0
    el_cy = lmks[36:42].mean(0)[1]
    er_cy = lmks[42:48].mean(0)[1]
    nose_y = lmks[27][1]
    bs["eyeLookUpLeft"]    = _norm((nose_y-el_cy)/fw, 0.0, 0.06)
    bs["eyeLookUpRight"]   = _norm((nose_y-er_cy)/fw, 0.0, 0.06)
    bs["eyeLookDownLeft"]  = _norm((el_cy-nose_y)/fw, 0.0, 0.06)
    bs["eyeLookDownRight"] = _norm((er_cy-nose_y)/fw, 0.0, 0.06)

    # ── 입 (jaw/mouth) ───────────────────────────────────────────
    # jawOpen: landmark 62(상순) ↔ 66(하순) 거리
    mouth_h = np.linalg.norm(lmks[62] - lmks[66])
    bs["jawOpen"] = _norm(mouth_h/fw, 0.0, 0.25)

    # jaw 좌우: 턱 끝 offset
    jaw_cx = (lmks[0][0]+lmks[16][0])/2
    chin_x = lmks[8][0]
    joff = (chin_x-jaw_cx)/fw
    bs["jawLeft"]    = _norm(-joff, 0.0, 0.08)
    bs["jawRight"]   = _norm( joff, 0.0, 0.08)
    bs["jawForward"] = 0.0  # [미구현] depth 없이 추정 불가

    # mouthSmile: 입꼬리(48,54) vs 입 중앙 높이
    mcy = (lmks[51][1]+lmks[57][1])/2
    sl = (mcy - lmks[48][1])/fw
    sr = (mcy - lmks[54][1])/fw
    bs["mouthSmileLeft"]  = _norm( sl, 0.0, 0.08)
    bs["mouthSmileRight"] = _norm( sr, 0.0, 0.08)
    bs["mouthFrownLeft"]  = _norm(-sl, 0.0, 0.08)
    bs["mouthFrownRight"] = _norm(-sr, 0.0, 0.08)

    # mouthPucker: 입 폭 감소
    mw = np.linalg.norm(lmks[48]-lmks[54])/fw
    bs["mouthPucker"]  = _norm(0.22-mw, 0.0, 0.15)
    bs["mouthFunnel"]  = bs["mouthPucker"]*0.7 + bs["jawOpen"]*0.3
    bs["mouthLeft"]    = bs["jawLeft"]
    bs["mouthRight"]   = bs["jawRight"]
    bs["mouthDimpleLeft"]   = bs["mouthSmileLeft"]  * 0.5
    bs["mouthDimpleRight"]  = bs["mouthSmileRight"] * 0.5
    bs["mouthStretchLeft"]  = bs["mouthSmileLeft"]  * 0.3
    bs["mouthStretchRight"] = bs["mouthSmileRight"] * 0.3
    bs["mouthPressLeft"]    = 0.0
    bs["mouthPressRight"]   = 0.0
    bs["mouthRollLower"]    = 0.0
    bs["mouthRollUpper"]    = 0.0
    bs["mouthShrugLower"]   = 0.0
    bs["mouthShrugUpper"]   = 0.0
    bs["mouthLowerDownLeft"]  = bs["jawOpen"] * 0.6
    bs["mouthLowerDownRight"] = bs["jawOpen"] * 0.6
    bs["mouthUpperUpLeft"]    = bs["jawOpen"] * 0.3
    bs["mouthUpperUpRight"]   = bs["jawOpen"] * 0.3
    bs["mouthClose"]     = _norm(1-bs["jawOpen"], 0.5, 1.0)
    bs["lipsUpperClose"] = bs["mouthClose"] * 0.5
    bs["lipsLowerClose"] = bs["mouthClose"] * 0.5

    # ── 눈썹 ─────────────────────────────────────────────────────
    brow_li = lmks[21];  brow_ri = lmks[22]
    eye_li  = lmks[39];  eye_ri  = lmks[42]
    d_l = (eye_li[1]-brow_li[1])/fw
    d_r = (eye_ri[1]-brow_ri[1])/fw
    bs["browInnerUp"]     = _norm(-(d_l+d_r)/2 + 0.18, 0.0, 0.06)
    bs["browOuterUpLeft"]  = _norm((lmks[39][1]-lmks[17][1])/fw-0.12, 0.0, 0.05)
    bs["browOuterUpRight"] = _norm((lmks[42][1]-lmks[26][1])/fw-0.12, 0.0, 0.05)
    bs["browDownLeft"]   = _norm(d_l - 0.20, 0.0, 0.07)
    bs["browDownRight"]  = _norm(d_r - 0.20, 0.0, 0.07)

    # ── 볼/코 ────────────────────────────────────────────────────
    bs["cheekPuff"]        = 0.0  # [미구현] 외곽 폭 변화 필요
    bs["cheekSquintLeft"]  = bs["mouthSmileLeft"]  * 0.6
    bs["cheekSquintRight"] = bs["mouthSmileRight"] * 0.6
    bs["noseSneerLeft"]    = bs["mouthSmileLeft"]  * 0.3

    return bs


def coeff_to_blendshapes_52(
    coeff: np.ndarray,
    lmks_crop: np.ndarray
) -> dict:
    """
    facemap_3dmm coeff + 68 landmark → 52 blendshape.

    [설계 결정]
    A) [공식] landmark 기하학 기반 (EAR, 거리 비율)
    B) [근사] expression PCA 첫 6차원을 보조 가중치로 추가
       → PC 부호와 의미는 학습 데이터에 의존, 검증 안 됨
    """
    c = split_coeff(coeff)
    expr_std = c["expression"] / (c["expression"].std() + 1e-6)

    # A) Landmark 기반
    bs = _lmk_blendshapes(lmks_crop)

    # B) Expression PCA 보조 [근사]
    boosts = {
        "jawOpen":         float(np.clip( expr_std[0]*0.08, 0, 0.25)),
        "mouthSmileLeft":  float(np.clip( expr_std[1]*0.04, 0, 0.15)),
        "mouthSmileRight": float(np.clip( expr_std[1]*0.04, 0, 0.15)),
        "eyeBlinkLeft":    float(np.clip( expr_std[2]*0.04, 0, 0.15)),
        "eyeBlinkRight":   float(np.clip( expr_std[2]*0.04, 0, 0.15)),
        "browInnerUp":     float(np.clip( expr_std[3]*0.04, 0, 0.15)),
    }
    for k, v in boosts.items():
        bs[k] = float(np.clip(bs.get(k, 0)+v, 0, 1))

    # 52개 순서 맞춰 반환
    return {n: float(np.clip(bs.get(n, 0.0), 0.0, 1.0)) for n in BLENDSHAPE_NAMES}


print("✅ 52 Blendshape 함수 정의 완료")
print(f"   blendshape 수: {len(BLENDSHAPE_NAMES)}")


## Cell 7 · 시각화 함수

In [ ]:
# ============================================================
# Cell 7: 시각화 유틸리티
# 중간 셀: plt.show()로 화면 표시만
# 저장: 마지막 셀에서만 수행
# ============================================================

def imshow(img_bgr, title="", figsize=(8,6)):
    """BGR ndarray → Colab 인라인 표시"""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb); plt.title(title); plt.axis("off")
    plt.tight_layout(); plt.show()


def draw_bbox(img_bgr, bbox, n_faces=1):
    """원본 이미지에 bbox 그리기"""
    vis = img_bgr.copy()
    x1,y1,x2,y2 = [int(v) for v in bbox]
    cv2.rectangle(vis, (x1,y1),(x2,y2),(0,255,0),3)
    cv2.putText(vis, f"Face (largest of {n_faces})",
                (x1, max(y1-10,10)), cv2.FONT_HERSHEY_SIMPLEX, 0.9,(0,255,0),2)
    return vis


def draw_landmarks(img_bgr, lmks, show_idx=True):
    """이미지에 68 landmark 그리기. show_idx=True면 index 번호 표시"""
    vis = img_bgr.copy()
    for i,(x,y) in enumerate(lmks):
        cv2.circle(vis,(int(x),int(y)),3,(0,0,255),-1)
        if show_idx:
            cv2.putText(vis, str(i),(int(x)+3,int(y)-3),
                        cv2.FONT_HERSHEY_PLAIN, 0.65,(255,220,0),1)
    return vis


def blendshape_bar_chart(bs_dict: dict) -> np.ndarray:
    """
    52 blendshape 막대그래프.
    plt.show()로 표시 + BGR numpy array 반환 (저장용).
    """
    names  = list(bs_dict.keys())
    values = list(bs_dict.values())
    colors = plt.cm.RdYlGn(values)

    fig, ax = plt.subplots(figsize=(10, 14))
    bars = ax.barh(range(len(names)), values, color=colors)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Weight (0–1)")
    ax.set_title(
        "52 Blendshape Weights\n"
        "[근사/규칙 기반 — Qualcomm→ARKit 공식 매핑 없음]",
        fontsize=10
    )
    ax.invert_yaxis()
    for bar,v in zip(bars,values):
        ax.text(min(v+0.02,0.94), bar.get_y()+bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=7)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=120)
    buf.seek(0)
    plt.show(); plt.close()

    arr = np.frombuffer(buf.read(), dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


print("✅ 시각화 함수 정의 완료")


## Cell 8 · 이미지 업로드 & InsightFace 얼굴 검출

In [ ]:
# ============================================================
# Cell 8: 이미지 업로드 → InsightFace bbox 검출 → 시각화
# ============================================================

from google.colab import files

print("얼굴이 포함된 이미지를 업로드하세요 (JPG / PNG):")
uploaded = files.upload()

fname = list(uploaded.keys())[0]
buf   = np.frombuffer(uploaded[fname], dtype=np.uint8)
img_bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
print(f"✅ 이미지 로드: {fname}  크기={img_bgr.shape}")

# InsightFace 검출
faces = face_app.get(img_bgr)
if not faces:
    raise RuntimeError("얼굴이 검출되지 않았습니다. 다른 이미지를 사용하세요.")
print(f"   검출된 얼굴 수: {len(faces)}")

# 가장 큰 얼굴 선택
def bbox_area(f):
    b = f.bbox
    return (b[2]-b[0])*(b[3]-b[1])

face   = max(faces, key=bbox_area)
bbox   = tuple(face.bbox.astype(int))
print(f"   선택 bbox : {bbox}  (면적={bbox_area(face):.0f}px²)")

# 결과 버퍼 저장
RESULTS["original_img"] = img_bgr.copy()
RESULTS["bbox"]         = bbox

# 시각화
bbox_vis = draw_bbox(img_bgr, bbox, len(faces))
imshow(bbox_vis, f"InsightFace 검출 ({len(faces)}개 얼굴 중 최대 선택)")


## Cell 9 · Crop 전처리 시각화

In [ ]:
# ============================================================
# Cell 9: InsightFace bbox → 128×128 crop 시각화
# (실제 추론 입력과 동일한 crop)
# ============================================================

CROP_SIZE = 128
img_tensor, crop_rgb = preprocess_for_facemap(img_bgr, bbox, CROP_SIZE)
crop_bgr = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR)

print(f"Crop shape  : {crop_rgb.shape}  (HWC uint8 RGB)")
print(f"Tensor shape: {img_tensor.shape}  (NCHW float32)")
print(f"Tensor range: [{img_tensor.min():.3f}, {img_tensor.max():.3f}]")

# 256×256으로 확대해 표시
imshow(
    cv2.resize(crop_bgr, (256,256), interpolation=cv2.INTER_NEAREST),
    "facemap_3dmm 입력 crop (128×128 → 256×256 확대)"
)


## Cell 10 · facemap_3dmm 추론 → coeff 출력

In [ ]:
# ============================================================
# Cell 10: Qualcomm facemap_3dmm PyTorch 로컬 추론
#
# [추론 흐름]
#   img_tensor (1,3,128,128 float32)
#     → facemap_model(img_tensor)
#     → coeff (265,) float32
#
# [AI Hub 불필요]
#   from_pretrained()은 PyTorch 가중치를 로컬에 다운로드하며
#   eval-mode=fp(PyTorch)로 로컬 CPU 추론합니다.
# ============================================================

print("추론 중...")
coeff = run_facemap(facemap_model, img_tensor)

print(f"✅ coeff shape: {coeff.shape}  dtype={coeff.dtype}")

c = split_coeff(coeff)
print()
print("─"*60)
print("  [identity  0:80 ] (shape PCA, 공식)")
print(f"    min={c["identity"].min():.4f}  max={c["identity"].max():.4f}"
      f"  std={c["identity"].std():.4f}")
print(f"    첫 5개: {c["identity"][:5]}")

print()
print("  [expression 80:144] (expression PCA, 공식)")
print(f"    min={c["expression"].min():.4f}  max={c["expression"].max():.4f}"
      f"  std={c["expression"].std():.4f}")
print(f"    첫 5개: {c["expression"][:5]}")

print()
print("  [rotation  144:147] Rodrigues 3D (공식)")
print(f"    {c["rotation"]}")

print()
print("  [translation 147:150] tx,ty,tz (공식)")
print(f"    {c["translation"]}")

print()
print("  [focal  150:151] (공식)")
print(f"    {c["focal"]}")

print()
print("  [unused 151:265] 텍스처/조명 114D (미사용)")
print(f"    min={c["unused"].min():.4f}  max={c["unused"].max():.4f}")
print("─"*60)

RESULTS["coeff"] = coeff


## Cell 11 · 68 Landmark 복원 & 시각화

In [ ]:
# ============================================================
# Cell 11: 3DMM coeff → 68 landmark 복원 & 시각화
#
# [좌표계 구분]
#  lmks_crop : crop 128×128 기준 픽셀 좌표
#  lmks_orig : 원본 이미지 기준 픽셀 좌표 (역변환)
#
# [bbox 일관성 보장]
#  crop 생성과 역변환 모두 동일한 InsightFace bbox 사용
# ============================================================

# crop 좌표계 landmark
print("3DMM landmark 복원 중...")
lmks_crop = reconstruct_landmarks68_from_coeff(coeff)
print(f"  lmks_crop shape: {lmks_crop.shape}")
print(f"  x: [{lmks_crop[:,0].min():.1f}, {lmks_crop[:,0].max():.1f}]")
print(f"  y: [{lmks_crop[:,1].min():.1f}, {lmks_crop[:,1].max():.1f}]")

# 원본 좌표 역변환
lmks_orig = crop_to_original(lmks_crop, bbox, CROP_SIZE)
print(f"  lmks_orig x: [{lmks_orig[:,0].min():.1f}, {lmks_orig[:,0].max():.1f}]")
print(f"  lmks_orig y: [{lmks_orig[:,1].min():.1f}, {lmks_orig[:,1].max():.1f}]")

# crop 위에 landmark 확인
crop_vis = cv2.resize(crop_bgr, (256,256))
scale_vis = 256/CROP_SIZE
for i,(x,y) in enumerate(lmks_crop):
    cv2.circle(crop_vis, (int(x*scale_vis), int(y*scale_vis)), 3, (0,0,255), -1)
imshow(crop_vis, "Crop 기준 Landmark (256×256 확대)")

# 원본 위 landmark + index 번호
lmk_vis = draw_landmarks(img_bgr, lmks_orig, show_idx=True)
imshow(lmk_vis, "원본 이미지 기준 68 Landmark (index 표시)", figsize=(10,8))

RESULTS["landmarks68"]      = lmks_orig
RESULTS["landmark_vis_img"] = lmk_vis
print("\n✅ Landmark 복원 완료")


## Cell 12 · 52 Blendshape 변환 & 시각화

In [ ]:
# ============================================================
# Cell 12: 52 Blendshape 변환 & 막대그래프 시각화
#
# [주의]
# Qualcomm facemap_3dmm → ARKit 52 blendshape 공식 매핑 없음.
# landmark 거리비율(EAR 등) + expression PCA 보조 신호 기반 근사.
# ============================================================

bs_52 = coeff_to_blendshapes_52(coeff, lmks_crop)

print("="*55)
print("52 Blendshape Values  (ARKit 표준 명칭, 0~1)")
print("[근사/규칙 기반 — 공식 매핑 아님]")
print("="*55)
for i,(k,v) in enumerate(bs_52.items()):
    bar = "█"*int(v*20)
    print(f"  {i:2d}. {k:<25} {v:.4f}  {bar}")

print("\n[필수 6개 항목]")
for k in ["jawOpen","eyeBlinkLeft","eyeBlinkRight",
          "mouthSmileLeft","mouthSmileRight","browInnerUp"]:
    print(f"  {k}: {bs_52[k]:.4f}")

# 막대그래프 시각화
print("\n막대그래프 생성 중...")
bs_vis = blendshape_bar_chart(bs_52)

RESULTS["blendshape_52"]      = bs_52
RESULTS["blendshape_vis_img"] = bs_vis
print("✅ Blendshape 변환 완료")


## Cell 13 (마지막) · 저장 & 다운로드
> **이 셀 실행 전까지 로컬에 파일이 생성되지 않습니다.**

In [ ]:
# ============================================================
# Cell 13 (마지막): 결과 저장 & 다운로드
#
# 저장 파일:
#   coeff.json            — 265D 3DMM coefficient
#   landmarks68.json      — 68 landmark 원본 좌표
#   blendshape_52.json    — 52 blendshape (three.js 바로 사용 가능)
#   landmark_vis.png      — landmark 시각화
#   blendshape_vis.png    — blendshape 막대그래프
#
# [three.js 연결 예시]
#   const d = await fetch("blendshape_52.json").then(r=>r.json());
#   Object.entries(d.blendshapes).forEach(([name,val])=>{
#     const idx = mesh.morphTargetDictionary[name];
#     if(idx!==undefined) mesh.morphTargetInfluences[idx]=val;
#   });
# ============================================================

from google.colab import files as colab_files

OUT = Path("/content/output")
OUT.mkdir(exist_ok=True)
saved = []

# 1) coeff.json
c = split_coeff(RESULTS["coeff"])
coeff_data = {
    "model":  "Qualcomm facemap_3dmm (qai-hub-models v0.49.1)",
    "inference_mode": "PyTorch fp32 local",
    "coeff_265": RESULTS["coeff"].tolist(),
    "split": {
        "identity":    c["identity"].tolist(),
        "expression":  c["expression"].tolist(),
        "rotation":    c["rotation"].tolist(),
        "translation": c["translation"].tolist(),
        "focal":       float(c["focal"][0]),
        "unused_note": "coeff[151:265] — 텍스처/조명, 후처리 미사용"
    }
}
p = OUT/"coeff.json"
p.write_text(json.dumps(coeff_data, indent=2))
saved.append(p); print(f"✅ {p.name}")

# 2) landmarks68.json
lmk_data = {
    "model": "InsightFace bbox → facemap_3dmm 3DMM 복원",
    "coordinate": "원본 이미지 픽셀 (x, y)",
    "index_standard": "dlib/FAN 68-point",
    "bbox_used": list(RESULTS["bbox"]),
    "landmarks": RESULTS["landmarks68"].tolist()
}
p = OUT/"landmarks68.json"
p.write_text(json.dumps(lmk_data, indent=2))
saved.append(p); print(f"✅ {p.name}")

# 3) blendshape_52.json (three.js / GLB 렌더러용)
bs_data = {
    "format": "ARKit/MediaPipe 52 Blendshapes",
    "source_model": "Qualcomm facemap_3dmm (qai-hub-models v0.49.1)",
    "mapping_method": "landmark geometry (EAR, distance ratio) + expression PCA",
    "official_mapping": False,
    "warning": (
        "Qualcomm→ARKit 공식 매핑 없음. landmark 거리비율 기반 근사입니다. "
        "Google Raccoon GLB morph target 이름이 다를 경우 key 수동 매핑 필요."
    ),
    "threejs_usage": (
        "mesh.morphTargetInfluences[mesh.morphTargetDictionary[name]] = value"
    ),
    "blendshapes": RESULTS["blendshape_52"]
}
p = OUT/"blendshape_52.json"
p.write_text(json.dumps(bs_data, indent=2, ensure_ascii=False))
saved.append(p); print(f"✅ {p.name}")

# 4) landmark_vis.png
p = OUT/"landmark_vis.png"
cv2.imwrite(str(p), RESULTS["landmark_vis_img"])
saved.append(p); print(f"✅ {p.name}")

# 5) blendshape_vis.png
p = OUT/"blendshape_vis.png"
cv2.imwrite(str(p), RESULTS["blendshape_vis_img"])
saved.append(p); print(f"✅ {p.name}")

print(f"\n총 {len(saved)}개 파일 저장 완료. 다운로드 시작...")
for fp in saved:
    colab_files.download(str(fp))
    print(f"  ↓ {fp.name}")

print("\n✅ 모든 다운로드 완료!")
print()
print("─"*60)
print("three.js / Google Raccoon GLB 연결 가이드")
print("─"*60)
print("""
import * as THREE from "three";
import { GLTFLoader } from "three/examples/jsm/loaders/GLTFLoader";

const loader = new GLTFLoader();
loader.load("raccoon.glb", (gltf) => {
  // mesh 이름은 실제 GLB에 맞게 수정하세요
  const mesh = gltf.scene.getObjectByName("Face");

  fetch("blendshape_52.json").then(r => r.json()).then(data => {
    Object.entries(data.blendshapes).forEach(([name, val]) => {
      const idx = mesh.morphTargetDictionary[name];
      if (idx !== undefined) mesh.morphTargetInfluences[idx] = val;
    });
  });
});

// ⚠️ morph target 이름이 ARKit과 다르면 blendshape_52.json의
//    key를 아바타 명칭에 맞게 수동 매핑하세요.
""")
